In [0]:
customers_df = spark.table("workspace.silver.customers")
products_df = spark.table("workspace.silver.products")
orders_df = spark.table("workspace.silver.orders")


In [0]:
sales_df = (
    orders_df
    .join(products_df, on="product_id", how="inner")
    .join(customers_df, on="customer_id", how="inner")
)

display(sales_df)

In [0]:
from pyspark.sql.functions import col

sales_df = sales_df.withColumn(
    "line_revenue",
    col("quantity") * col("unit_price")
)

In [0]:
sales_df.select(
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_revenue"
).show()

In [0]:
from pyspark.sql.functions import year, month, sum

monthly_revenue_df = (
    sales_df
    .filter(col("order_status") == "Completed")
    .groupBy(
        year("order_date").alias("year"),
        month("order_date").alias("month")
    )
    .agg(sum("line_revenue").alias("total_revenue"))
    .orderBy("year", "month")
)

display(monthly_revenue_df)

In [0]:
product_revenue_df = (
    sales_df
    .filter(col("order_status") == "Completed")
    .groupBy("product_id", "product_name")
    .agg(sum("line_revenue").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)

display(product_revenue_df)

In [0]:
country_revenue_df = (
    sales_df
    .filter(col("order_status") == "Completed")
    .groupBy("country")
    .agg(sum("line_revenue").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)

display(country_revenue_df)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

In [0]:
(
    monthly_revenue_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.monthly_revenue")
)

In [0]:
(
    product_revenue_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.product_revenue")
)

In [0]:
(  
    country_revenue_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.country_revenue")
)

In [0]:
display(spark.sql("SHOW TABLES IN workspace.gold"))

In [0]:
from pyspark.sql.functions import avg, sum

order_revenue_df = (
    sales_df
    .filter(col("order_status") == "Completed")
    .groupBy("order_id")
    .agg(sum("line_revenue").alias("order_revenue"))
)

average_order_value_df = order_revenue_df.agg(
    avg("order_revenue").alias("average_order_value")
)

display(average_order_value_df)

In [0]:
(
    average_order_value_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.average_order_value")
)

In [0]:
top_customers_df = (
    sales_df
    .filter(col("order_status") == "Completed")
    .groupBy("customer_id", "customer_name")
    .agg(sum("line_revenue").alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)

display(top_customers_df)

In [0]:
(
    top_customers_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.gold.top_customers")
)

In [0]:
display(spark.sql("SHOW TABLES IN workspace.gold"))